# Full eight-selector reliability experiment

This notebook is the only new experiment code. It uses official MedMNIST train as D and official test as E. Selection is performed once on clean D; the same final checkpoint is evaluated on clean E and MedMNIST-C E. No validation split is used for selection, checkpoint choice, calibration, or threshold choice.

The run is controlled by environment variables so the same notebook supports smoke, first-pass, and confirmation runs. It writes resumable artifacts: selected indices, checkpoints, per-sample prediction NPZ files, and metric CSV/JSONL files.

In [ ]:
from pathlib import Path
import os, sys, json, time, random, math, csv, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from PIL import Image

# ---------- paths and protocol ----------
PREP = Path(os.environ.get('PREP_ROOT', Path.cwd())).resolve()
GRAPH_ROOT = PREP / 'repos' / 'graphcov'
MEDC_ROOT = PREP / 'repos' / 'medmnistc'
DATA_ROOT = Path(os.environ.get('MEDMNIST_ROOT', '/root/autodl-tmp/NCFM_F2_SOBOL_PAIRED_PATH_20260812/assets/datasets/medmnist')).resolve()
CORR_ROOT = Path(os.environ.get('MEDMNISTC_ROOT', str(PREP / 'medmnistc_data'))).resolve()
OUT = Path(os.environ.get('RELIABILITY_OUT', str(PREP / 'full_run_outputs'))).resolve()
CACHE = Path(os.environ.get('GRAPH_CACHE', str(PREP / 'graph_cache'))).resolve()
DATASETS = [x.strip().lower() for x in os.environ.get('DATASETS', 'pathmnist,organsmnist').split(',') if x.strip()]
METHODS = ['random', 'el2n_top', 'forgetting', 'eva', 'facility', 'fps', 'herding', 'graph_a2']
RATIOS = [float(x) for x in os.environ.get('RATIOS', '0.02,0.05').split(',')]
SEEDS = [int(x) for x in os.environ.get('SEEDS', '0,1,2').split(',')]
EPOCHS = int(os.environ.get('EPOCHS', '1000'))
DYNAMICS_EPOCHS = int(os.environ.get('DYNAMICS_EPOCHS', '200'))
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '256'))
NUM_WORKERS = int(os.environ.get('NUM_WORKERS', '4'))
EMBEDDING_SOURCE = os.environ.get('EMBEDDING_SOURCE', 'uni')
AUGMENT = os.environ.get('AUGMENT', '0') == '1'
SIZE = int(os.environ.get('IMAGE_SIZE', '224'))
SMOKE_N = int(os.environ.get('SMOKE_N', '0'))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT.mkdir(parents=True, exist_ok=True); CACHE.mkdir(parents=True, exist_ok=True); CORR_ROOT.mkdir(parents=True, exist_ok=True)
print({'PREP': str(PREP), 'DATA_ROOT': str(DATA_ROOT), 'CORR_ROOT': str(CORR_ROOT), 'OUT': str(OUT), 'device': str(DEVICE), 'datasets': DATASETS, 'methods': METHODS, 'ratios': RATIOS, 'seeds': SEEDS, 'epochs': EPOCHS, 'augment': AUGMENT})

# ---------- import pinned upstream code ----------
sys.path.insert(0, str(GRAPH_ROOT))
sys.path.insert(0, str(MEDC_ROOT))
from medmnist import INFO
from graphcov.run.data import get_transform, get_train_transform, AugmentedDataset, get_labels
from graphcov.run.embeddings import load_or_compute_embeddings, load_or_compute_raw_dynamics
from graphcov.run.eva import get_optimal_windows, derive_eva_scores
from graphcov.run.selection import select, get_available_methods
from graphcov.run.embeddings import ResNet18WithFeatures
from graphcov.run.evaluation import train_one_epoch, set_seed
from medmnistc.dataset_manager import DatasetManager
from medmnistc.dataset import CorruptedMedMNIST
from medmnistc.corruptions.registry import CORRUPTIONS_DS, CORRUPTIONS_DS_FOLDS, DATASET_RGB

assert [m for m in METHODS if m not in get_available_methods()] == [], get_available_methods()
print('Pinned GraphCov methods:', METHODS)

# ---------- data ----------
def load_med(name, split, size=SIZE):
    info = INFO[name]
    cls = getattr(__import__('medmnist', fromlist=[info['python_class']]), info['python_class'])
    ds = cls(split=split, transform=get_transform(info['n_channels'], size), download=True, size=size, root=str(DATA_ROOT))
    return ds, info

def flat_labels(ds):
    y = np.asarray(ds.labels).reshape(-1).astype(np.int64)
    return y

def maybe_limit(ds, y, n):
    if n <= 0 or n >= len(ds): return ds, y
    # Keep a deterministic prefix only for smoke testing; full runs use all samples.
    idx = np.arange(n, dtype=np.int64)
    return Subset(ds, idx.tolist()), y[idx]

def corruption_names(name):
    available = set(CORRUPTIONS_DS[name])
    groups = {'noise': ['gaussian_noise','speckle_noise','impulse_noise','shot_noise'], 'blur': ['gaussian_blur','defocus_blur','motion_blur','zoom_blur'], 'brightness': ['brightness_down','brightness_up']}
    chosen = []
    for group, candidates in groups.items():
        hit = next((x for x in candidates if x in available), None)
        if hit is not None: chosen.append((group, hit))
    return chosen

def ensure_corruptions(name):
    wanted = [c for _, c in corruption_names(name)]
    missing = [c for c in wanted if not (CORR_ROOT / name / f'{c}.npz').exists()]
    if missing:
        print('Generating missing MedMNIST-C files:', name, missing)
        DatasetManager(medmnist_path=str(DATA_ROOT), output_path=str(CORR_ROOT), random_seed=0).create_dataset(name)
    return wanted

# ---------- metrics: all computed from per-sample logits ----------
def probs_from_logits(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    p = np.exp(z); return p / p.sum(axis=1, keepdims=True)

def ece_top(y, p, bins=15):
    pred = p.argmax(1); conf = p.max(1); out = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins, endpoint=False), np.linspace(0, 1, bins + 1)[1:]):
        mask = (conf > lo) & (conf <= hi if hi < 1 else conf <= hi)
        if mask.any(): out += mask.mean() * abs((pred[mask] == y[mask]).mean() - conf[mask].mean())
    return float(out)

def risk_coverage(y, p, accept_fraction=0.80):
    conf = p.max(1); order = np.argsort(-conf, kind='stable'); n = max(1, int(round(len(y) * accept_fraction)))
    accepted = order[:n]; risk = float(np.mean(p.argmax(1)[accepted] != y[accepted]))
    errors = (p.argmax(1)[order] != y[order]).astype(np.float64)
    aurc = float(np.mean(np.cumsum(errors) / np.arange(1, len(y) + 1)))
    return risk, aurc

def metric_row(y, logits, prefix=''):
    y = np.asarray(y, dtype=np.int64).reshape(-1); logits = np.asarray(logits, dtype=np.float32); p = probs_from_logits(logits); pred = p.argmax(1)
    recalls = {int(c): float(np.mean(pred[y == c] == c)) if np.any(y == c) else float('nan') for c in np.unique(y)}
    risk80, aurc = risk_coverage(y, p)
    true_p = np.clip(p[np.arange(len(y)), y], 1e-12, 1.0)
    onehot = np.eye(p.shape[1], dtype=np.float32)[y]
    high_conf_errors = int(np.sum((p.max(1) >= 0.80) & (pred != y)))
    row = {prefix+'acc': float(np.mean(pred == y)), prefix+'ba': float(np.mean(list(recalls.values()))), prefix+'worst_recall': float(np.nanmin(list(recalls.values()))), prefix+'nll': float(-np.mean(np.log(true_p))), prefix+'brier': float(np.mean(np.sum((p-onehot)**2, axis=1))), prefix+'ece15': ece_top(y,p), prefix+'risk_at_80': risk80, prefix+'aurc': aurc, prefix+'high_conf_errors': high_conf_errors}
    for c, v in recalls.items(): row[f'{prefix}recall_{c}'] = v
    return row, p, pred

def save_predictions(path, sample_id, y, logits, probs=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    if probs is None: probs = probs_from_logits(logits)
    np.savez_compressed(path, sample_id=np.asarray(sample_id), y_true=np.asarray(y), logits=np.asarray(logits, dtype=np.float32), probs=np.asarray(probs, dtype=np.float32))

# ---------- selection dependencies ----------
def get_selection_data(name, train_ds, labels, info, seed):
    needs_embedding = any(m in {'facility','fps','herding','graph_a2'} for m in METHODS)
    emb = None
    if needs_embedding:
        emb = load_or_compute_embeddings(name, 'train', EMBEDDING_SOURCE, train_ds, len(info['label']), info['n_channels'], size=SIZE, seed=seed, cache_dir=CACHE, verbose=True)['embeddings']
    dynamic = None
    if any(m in {'el2n_top','forgetting','eva'} for m in METHODS):
        dyn_ds, _ = load_med(name, 'train', size=28)
        dynamic = load_or_compute_raw_dynamics(name, 'train', dyn_ds, len(info['label']), info['n_channels'], size=28, seed=seed, eva_epochs=DYNAMICS_EPOCHS, window_size=10, cache_dir=CACHE, verbose=True)
    return emb, dynamic

def choose(name, labels, info, embeddings, dynamic, ratio, seed):
    n_classes = len(info['label']); budget = int(len(labels) * ratio) // n_classes
    kwargs = dict(method='random', labels=labels, budget_per_class=budget, seed=seed, verbose=False)
    selected = {}
    for method in METHODS:
        args = dict(kwargs, method=method, embeddings=embeddings)
        if dynamic is not None:
            all_l2 = dynamic['all_l2_scores']; el2n = all_l2[:min(20, len(all_l2))].mean(0)
            args.update(el2n_scores=el2n, forgetting_scores=dynamic['forgetting_scores'])
            if method == 'eva':
                early, late = get_optimal_windows(name, ratio, eva_epochs=DYNAMICS_EPOCHS, verbose=False)
                args['eva_scores'] = derive_eva_scores(all_l2, window_size=10, early_window_start=early, late_window_start=late, verbose=False)[0]
        if method in {'facility','fps','herding','graph_a2'}:
            args['importance'] = np.ones(len(labels), dtype=np.float32)
        if method == 'graph_a2': args.update(global_selection=True, k_neighbors=50, k_hops=2)
        selected[method] = np.asarray(select(**args), dtype=np.int64)
        if len(np.unique(selected[method])) != len(selected[method]): raise RuntimeError(f'duplicate selection: {method}')
    selected['full_train'] = np.arange(len(labels), dtype=np.int64)
    return selected, budget

# ---------- downstream training and prediction ----------
def train_model(train_ds, selected, test_ds, num_classes, in_channels, seed):
    set_seed(seed, deterministic=False)
    ds = train_ds
    if AUGMENT: ds = AugmentedDataset(ds, get_train_transform(in_channels, SIZE))
    loader = DataLoader(Subset(ds, selected.tolist()), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
    model = ResNet18WithFeatures(num_classes, in_channels, pretrained=False).to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0005)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss()
    for epoch in range(EPOCHS):
        train_one_epoch(model, loader, opt, crit)
        sch.step()
        if (epoch + 1) % max(1, EPOCHS // 10) == 0: print(f'epoch {epoch+1}/{EPOCHS}')
    return model

def predict(model, ds):
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    logits, ys = [], []
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            logits.append(model(x.to(DEVICE)).cpu().numpy()); ys.append(np.asarray(y).reshape(-1))
    return np.concatenate(ys), np.concatenate(logits)

def write_jsonl(path, row):
    with path.open('a', encoding='utf-8') as f: f.write(json.dumps(row, ensure_ascii=True) + '\n')

# ---------- resumable experiment ----------
summary_path = OUT / 'metrics.jsonl'
completed = set()
if summary_path.exists():
    for line in summary_path.read_text(encoding='utf-8').splitlines():
        try:
            r = json.loads(line); completed.add((r['dataset'], r['method'], r['ratio'], r['seed'], r['augment']))
        except Exception: pass
all_rows = []
for name in DATASETS:
    print('==========', name, '==========')
    train_ds, info = load_med(name, 'train', SIZE); test_ds, _ = load_med(name, 'test', SIZE)
    y_train = flat_labels(train_ds); y_test = flat_labels(test_ds)
    if SMOKE_N > 0: train_ds, y_train = maybe_limit(train_ds, y_train, SMOKE_N); test_ds, y_test = maybe_limit(test_ds, y_test, min(SMOKE_N, len(test_ds)))
    corr_files = ensure_corruptions(name)
    corr_ds = {c: CorruptedMedMNIST(name, c, norm_mean=[0.5] * info['n_channels'], norm_std=[0.5] * info['n_channels'], root=str(CORR_ROOT), as_rgb=True, mmap_mode='r') for c in corr_files}
    for ratio in RATIOS:
        for seed in SEEDS:
            embeddings, dynamic = get_selection_data(name, train_ds, y_train, info, seed)
            chosen, budget = choose(name, y_train, info, embeddings, dynamic, ratio, seed)
            for method, selected in chosen.items():
                key = (name, method, ratio, seed, int(AUGMENT))
                if key in completed: print('skip completed', key); continue
                tag = f'{name}__{method}__r{ratio:g}__s{seed}__aug{int(AUGMENT)}'
                run_dir = OUT / tag; run_dir.mkdir(parents=True, exist_ok=True)
                np.save(run_dir / 'selected_indices.npy', selected)
                np.savez_compressed(run_dir / 'selection_manifest.npz', selected_indices=selected, train_labels=y_train[selected], all_train_labels=y_train)
                print('training', tag, 'n=', len(selected), 'budget_per_class=', budget)
                model = train_model(train_ds, selected, test_ds, len(info['label']), info['n_channels'], seed)
                torch.save({'state_dict': model.state_dict(), 'dataset': name, 'method': method, 'ratio': ratio, 'seed': seed, 'augment': AUGMENT, 'size': SIZE}, run_dir / 'final.pt')
                y, logits = predict(model, test_ds); row, p, pred = metric_row(y, logits)
                save_predictions(run_dir / 'predictions_clean.npz', np.arange(len(y)), y, logits, p)
                base = {'dataset': name, 'method': method, 'ratio': ratio, 'seed': seed, 'augment': int(AUGMENT), 'n_selected': int(len(selected)), 'budget_per_class': int(budget), 'corruption': 'clean'}; base.update(row); write_jsonl(summary_path, base); all_rows.append(base)
                for cname, cds in corr_ds.items():
                    cy, clogits = predict(model, cds); n_clean = len(test_ds)
                    # MedMNIST-C stores five consecutive severity blocks. Save both full and per-severity evidence.
                    save_predictions(run_dir / f'predictions_{cname}.npz', np.arange(len(cy)), cy, clogits)
                    for severity in range(5):
                        sl = slice(severity * n_clean, (severity + 1) * n_clean)
                        crow, _, _ = metric_row(cy[sl], clogits[sl])
                        cbase = {'dataset': name, 'method': method, 'ratio': ratio, 'seed': seed, 'augment': int(AUGMENT), 'n_selected': int(len(selected)), 'budget_per_class': int(budget), 'corruption': cname, 'severity': severity + 1}
                        cbase.update(crow); cbase['clean_ba_drop'] = float(row['ba'] - crow['ba']); cbase['clean_worst_recall_drop'] = float(row['worst_recall'] - crow['worst_recall']); write_jsonl(summary_path, cbase); all_rows.append(cbase)
                del model
                if torch.cuda.is_available(): torch.cuda.empty_cache()

# Stable tabular export after every run; JSONL remains the append-only source of truth.
if summary_path.exists():
    df = pd.read_json(summary_path, lines=True)
    df.to_csv(OUT / 'metrics.csv', index=False)
    clean = df[df.corruption == 'clean']; clean.to_csv(OUT / 'clean_metrics.csv', index=False)
    corrupt = df[df.corruption != 'clean']; corrupt.to_csv(OUT / 'corruption_metrics_by_severity.csv', index=False)
    print('completed rows:', len(df), 'output:', OUT)
else:
    print('No rows written')
